## 1. Environment Setup & Dependencies
Install dependencies for Ultralytics YOLO, PyCOCOtools, Weighted Boxes Fusion (`ensemble-boxes`), and C++ accelerated evaluation (`faster-coco-eval`).

In [1]:
!pip install -q -U ultralytics pycocotools ensemble-boxes faster-coco-eval

## 2. Workspace & Data Setup
Mount Google Drive to persist checkpoints and serialized detections (`.npy`) across sessions, and download the VisDrone competition dataset to local storage.

In [2]:
import glob, os, pathlib, subprocess
# ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if ON_KAGGLE:
    RAW = pathlib.Path(glob.glob("/kaggle/input/**/train/annotations_coco.json", recursive=True)[0]).parents[1]
    WORK, OUT = "/kaggle/working", "/kaggle/working/outputs"
else:  # Colab: data from the Kaggle API, outputs on Drive
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
    if "KAGGLE_API_TOKEN" not in os.environ and "KAGGLE_KEY" not in os.environ:
        try: os.environ["KAGGLE_API_TOKEN"] = userdata.get("kaggle")  # one secret named `kaggle` holding the KGAT_... token
        except Exception: pass
    RAW, WORK, OUT = pathlib.Path("/content/raw"), "/content", "/content/drive/MyDrive/hw2_runs"
    if not (RAW / "train/annotations_coco.json").exists():
        subprocess.run("mkdir -p /content/dl /content/raw && cd /content/dl && pip install -q kaggle && "
                       "kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026 && "
                       "unzip -q -o *.zip -d /content/raw && rm -f *.zip", shell=True, check=True)
os.makedirs(OUT, exist_ok=True)
os.environ["HW2_DATA"] = str(RAW); os.environ["PYTHONPATH"] = "src"
print("data:", RAW, "| work:", WORK, "| outputs:", OUT)

Mounted at /content/drive
data: /content/raw | work: /content | outputs: /content/drive/MyDrive/hw2_runs


## 3. Core Source Modules
Write self-contained Python modules for data preparation, ignore region masking, standardized COCO evaluation, and multi-model ensembling.

In [3]:
!mkdir -p src

### `src/common.py`
Shared constants and dataset loaders. Loads COCO annotations and deduplicates `iscrowd=1` ignore regions.

In [4]:
%%writefile src/common.py
"""Shared helpers: dataset paths and COCO-json -> DataFrame loading."""
import json
import os
from pathlib import Path

import pandas as pd

# override with HW2_DATA=/path/to/raw when running on another machine
DATA = Path(os.environ.get("HW2_DATA", Path(__file__).resolve().parents[1] / "data" / "raw"))
CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]


def load_coco(split):
    """Return (images_df, objects_df, ignore_df) for 'train' or 'val'.

    Ignore regions are stored in the COCO json as iscrowd=1 boxes copied once per
    category (4 regions x 8 classes), so we dedupe them to one row per region.
    objects_df: real objects only (iscrowd=0), category 0..7 (0-based).
    """
    j = json.load(open(DATA / split / "annotations_coco.json"))
    imgs = pd.DataFrame(j["images"]).rename(columns={"id": "image_id"})
    ann = pd.DataFrame(j["annotations"])
    ann[["x", "y", "w", "h"]] = pd.DataFrame(ann.bbox.tolist(), index=ann.index)
    ann = ann.drop(columns=["bbox", "area"]).merge(
        imgs[["image_id", "video", "width", "height"]], on="image_id")
    objs = ann[ann.iscrowd == 0].copy()
    objs["cls"] = objs.category_id - 1
    ign = (ann[ann.iscrowd == 1]
           .drop_duplicates(["image_id", "x", "y", "w", "h"])
           [["image_id", "video", "x", "y", "w", "h", "width", "height"]])
    return imgs, objs, ign

Writing src/common.py


### `src/coco_utils.py`
Standardized evaluation protocol across all model families. Uses `faster-coco-eval` (C++ backend) to score mAP@[0.5:0.95], AP50, and small/medium/large scale breakdowns in seconds.

In [5]:
%%writefile src/coco_utils.py
"""COCO-style scoring shared by every detector family (Ultralytics models in eval_coco.py, torchvision in train_tv.py).

Uses faster_coco_eval (C++ drop-in replacement for pycocotools) when available so full-val scoring takes ~3s instead of ~10m.
"""
import json
from pathlib import Path

import numpy as np
try:
    import faster_coco_eval
    faster_coco_eval.init_as_pycocotools()
except Exception:
    pass
from pycocotools.cocoeval import COCOeval

from common import CLASSES


def _mean(x):
    return float(np.mean(x[x > -1]))


def score(gt, dets, img_ids, tag, ms_per_frame, out_dir="runs", **meta):
    """dets: list of dict(image_id, category_id (1-based), score, bbox [x, y, w, h] in source pixels)."""
    out = Path(out_dir); out.mkdir(exist_ok=True)
    np.save(out / f"dets_{tag}.npy", np.array([[d["image_id"], d["category_id"], d["score"], *d["bbox"]] for d in dets], np.float32))
    dt = gt.loadRes(dets)
    E = COCOeval(gt, dt, "bbox")
    E.params.imgIds, E.params.maxDets = img_ids, [1, 10, 500]
    E.evaluate(); E.accumulate(); E.summarize()
    st, P = E.stats, E.eval["precision"][:, :, :, 0, 2]
    st[0] = _mean(P)
    E2 = COCOeval(gt, dt, "bbox")
    E2.params.imgIds, E2.params.maxDets = [i for i in img_ids if gt.imgs[i]["width"] < 3000], [1, 10, 500]
    E2.evaluate(); E2.accumulate()
    res = dict(tag=tag, **meta, frames=len(img_ids), AP=st[0], AP_no4k=_mean(E2.eval["precision"][:, :, :, 0, 2]),
               AP50=st[1], AP75=st[2], AP_small=st[3], AP_medium=st[4], AP_large=st[5], AR500=st[8],
               ms_per_frame=ms_per_frame, per_class_AP={n: _mean(P[:, :, k]) for k, n in enumerate(CLASSES)})
    print(json.dumps(res, indent=1))
    json.dump(res, open(out / f"eval_{tag}.json", "w"), indent=1)
    return res

Writing src/coco_utils.py


### `src/prepare.py`
Builds Ultralytics-compatible dataset lists. Subsamples training sequences with a stride of 3 (`stride=3`) to eliminate adjacent frame redundancy while maintaining full video coverage.

In [6]:
%%writefile src/prepare.py
"""Build Ultralytics-ready lists + data.yaml (the provided layout images/<video>/N.jpg <-> labels/<video>/N.txt
already matches the YOLO convention, so no label conversion is needed).

- Split: official `train` (56 videos) / `val` (7 videos); no video appears in both, so it is a sequence-level split.
- Frame subsampling: keep every STRIDE-th frame of train (neighbouring frames are near-duplicates).
- Ignore regions are NOT in the YOLO labels; they are only used at evaluation (COCO iscrowd=1, see common.load_coco).

Usage: python src/prepare.py [--stride 3] [--out data/yolo]
"""
import argparse
from pathlib import Path

from common import CLASSES, DATA

ap = argparse.ArgumentParser()
ap.add_argument("--stride", type=int, default=3)
ap.add_argument("--out", default=str(DATA.parent / "yolo"))
a = ap.parse_args()
out = Path(a.out); out.mkdir(parents=True, exist_ok=True)


def frames(split, stride):
    keep = []
    for vid in sorted((DATA / split / "images").iterdir()):
        fr = sorted(vid.glob("*.jpg"))
        keep += fr[::stride]  # same phase for every video -> deterministic
    return keep


tr, va, va4 = frames("train", a.stride), frames("val", 1), frames("val", 4)  # val4: cheap per-epoch curves during training
(out / f"train_s{a.stride}.txt").write_text("\n".join(map(str, tr)))
(out / "val.txt").write_text("\n".join(map(str, va)))
(out / "val_s4.txt").write_text("\n".join(map(str, va4)))
(out / f"data_s{a.stride}.yaml").write_text(
    f"path: {out}\ntrain: train_s{a.stride}.txt\nval: val_s4.txt\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)))
print(f"train {len(tr)} frames (stride {a.stride}), val {len(va)} frames -> {out}")

Writing src/prepare.py


### `src/mask_ignore.py`
VisDrone marks dense crowds and distant parking lots as ignore regions (`iscrowd=1`). Standard training treats unannotated objects in these regions as negative background. This module paints ignore regions gray (114) and drops overlapping labels so models do not learn to penalize valid detections.

In [7]:
%%writefile src/mask_ignore.py
"""Make a train set where ignore regions are "don't care": paint them gray and drop labels that sit inside them.

Fast reuse: when building `yolo_masked_tv` (--splits train val) after `yolo_masked`, already-masked `train` images/labels
are symlinked instead of re-encoding 6,053 JPEGs a second time.
"""
import argparse
from multiprocessing import Pool
from pathlib import Path

from PIL import Image, ImageDraw

from common import CLASSES, DATA

ap = argparse.ArgumentParser()
ap.add_argument("--stride", type=int, default=3)
ap.add_argument("--out", default=str(DATA.parent / "yolo_masked"))
ap.add_argument("--reuse", default="", help="existing masked dir to symlink already-processed train frames from")
ap.add_argument("--splits", nargs="+", default=["train"], choices=["train", "val"], help="add val for the final train+val model")
a = ap.parse_args()
OUT = Path(a.out).resolve()
REUSE = Path(a.reuse).resolve() if a.reuse else None


def regions(p):
    if not p.exists():
        return []
    return [(cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
            for cx, cy, w, h in (map(float, l.split()) for l in p.read_text().splitlines() if l.strip())]


def inside_frac(b, r):
    ix = max(0.0, min(b[2], r[2]) - max(b[0], r[0])); iy = max(0.0, min(b[3], r[3]) - max(b[1], r[1]))
    return ix * iy / max((b[2] - b[0]) * (b[3] - b[1]), 1e-12)


def work(item):
    split, img = item
    vid, name = img.parent.name, img.stem
    dst_img, dst_lab = OUT / "images" / vid / img.name, OUT / "labels" / vid / f"{name}.txt"
    dst_img.parent.mkdir(parents=True, exist_ok=True); dst_lab.parent.mkdir(parents=True, exist_ok=True)
    if REUSE and (REUSE / "images" / vid / img.name).exists() and (REUSE / "labels" / vid / f"{name}.txt").exists():
        if not (dst_img.exists() or dst_img.is_symlink()):
            dst_img.symlink_to(REUSE / "images" / vid / img.name)
        if not (dst_lab.exists() or dst_lab.is_symlink()):
            dst_lab.symlink_to(REUSE / "labels" / vid / f"{name}.txt")
        return 0, False
    regs = regions(DATA / split / "ignore" / vid / f"{name}.txt")
    src_lab = DATA / split / "labels" / vid / f"{name}.txt"
    lines, kept = (src_lab.read_text().splitlines() if src_lab.exists() else []), []
    for l in lines:
        c, cx, cy, w, h = l.split(); cx, cy, w, h = map(float, (cx, cy, w, h))
        b = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
        if not any(inside_frac(b, r) > 0.5 for r in regs):
            kept.append(l)
    dst_lab.write_text("\n".join(kept))
    if dst_img.exists() or dst_img.is_symlink():
        dst_img.unlink()
    if regs:
        im = Image.open(img).convert("RGB"); W, H = im.size; d = ImageDraw.Draw(im)
        for x0, y0, x1, y1 in regs:
            d.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], fill=(114, 114, 114))
        im.save(dst_img, quality=95)
    else:
        dst_img.symlink_to(img)
    return len(lines) - len(kept), bool(regs)


if __name__ == "__main__":
    items = []
    for split in a.splits:
        for vid in sorted((DATA / split / "images").iterdir()):
            items += [(split, f) for f in sorted(vid.glob("*.jpg"))[::a.stride]]
    with Pool() as p:
        res = p.map(work, items, chunksize=32)
    name = "train" if a.splits == ["train"] else "trainval"
    (OUT / f"{name}_s{a.stride}.txt").write_text("\n".join(str(OUT / "images" / i.parent.name / i.name) for _, i in items))
    (OUT / f"data{'' if name == 'train' else '_trainval'}_s{a.stride}.yaml").write_text(
        f"path: {OUT}\ntrain: {name}_s{a.stride}.txt\nval: {OUT.parent / 'yolo' / 'val_s4.txt'}\nnames:\n"
        + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)))
    print(f"{len(items)} frames ({'+'.join(a.splits)}), {sum(r[1] for r in res)} newly masked, "
          f"{sum(r[0] for r in res)} labels dropped -> {OUT}")

Writing src/mask_ignore.py


### `src/eval_coco.py`
Runs inference on validation sequences for Ultralytics models, supporting multi-label NMS and test-time augmentation (TTA), and computes ignore-aware COCO metrics.

In [8]:
%%writefile src/eval_coco.py
"""COCO-style evaluation of an Ultralytics model on `val` that honours ignore regions (iscrowd=1 in the GT json).

Ultralytics' own val does not know about ignore regions, so false positives inside them are counted against the model;
the Kaggle scorer ignores them. This script is the project's single evaluation protocol for every experiment:
AP (0.5:0.95), AP50, AP75, AP_small/medium/large (COCO bins on source-pixel area), AR@500, per-class AP, speed.

Usage: python src/eval_coco.py --weights runs/x/weights/best.pt --imgsz 1280 --tag smoke
"""
import argparse
import time

from pycocotools.coco import COCO
from ultralytics import RTDETR, YOLO

from coco_utils import score
from common import DATA

ap = argparse.ArgumentParser()
ap.add_argument("--weights", required=True)
ap.add_argument("--imgsz", type=int, default=1280)
ap.add_argument("--conf", type=float, default=0.001)
ap.add_argument("--iou", type=float, default=0.6)     # NMS IoU
ap.add_argument("--split", default="val")
ap.add_argument("--tag", default="eval")
ap.add_argument("--bs", type=int, default=16, help="inference chunk size; lower it for heavy models / small GPUs")
ap.add_argument("--multi-label", action="store_true", help="NMS keeps several classes per box (as Ultralytics val does; predict() default keeps only the argmax class)")
ap.add_argument("--tta", action="store_true", help="Ultralytics test-time augmentation (flip + 3 scales); YOLO only")
ap.add_argument("--arch", default="yolo", choices=["yolo", "rtdetr"])
ap.add_argument("--limit", type=int, default=0, help="debug: evaluate only the first N frames")
a = ap.parse_args()
if a.multi_label:  # predict() calls nms.non_max_suppression without multi_label; val() passes multi_label=True
    import functools
    from ultralytics.utils import nms
    nms.non_max_suppression = functools.partial(nms.non_max_suppression, multi_label=True)

gt = COCO(str(DATA / a.split / "annotations_coco.json"))
img_ids = sorted(gt.getImgIds())[: a.limit or None]
files = [str(DATA / a.split / gt.imgs[i]["file_name"]) for i in img_ids]
model, B, dets = (RTDETR if a.arch == "rtdetr" else YOLO)(a.weights), a.bs, []  # chunks of 16: passing every path at once -> CUDA OOM

t0 = time.time()
for s in range(0, len(files), B):
    for r, iid in zip(model.predict(files[s:s + B], imgsz=a.imgsz, conf=a.conf, iou=a.iou, max_det=500, batch=B, augment=a.tta,
                                    verbose=False), img_ids[s:s + B]):
        xyxy, sc, c = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()
        dets += [dict(image_id=iid, category_id=int(k) + 1, score=float(p),
                      bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)])
                 for (x1, y1, x2, y2), p, k in zip(xyxy, sc, c)]
ms = (time.time() - t0) / len(files) * 1000  # end-to-end per frame (decode + preprocess + model + NMS)
score(gt, dets, img_ids, a.tag, ms, weights=a.weights, imgsz=a.imgsz, conf=a.conf, nms_iou=a.iou)

Writing src/eval_coco.py


### `src/train_tv.py`
Trains and evaluates torchvision detectors (Faster R-CNN, RetinaNet, FCOS) under the identical protocol and input lists for direct cross-architecture comparison.

In [9]:
%%writefile src/train_tv.py
"""Train + evaluate torchvision detectors under the same protocol as the YOLO runs.

  frcnn  : Faster R-CNN R50-FPN v2   (two-stage, anchor-based)
  retina : RetinaNet  R50-FPN v2     (one-stage, anchor-based)
  fcos   : FCOS       R50-FPN        (one-stage, anchor-free)
All start from COCO-pretrained weights (new 8-class heads), train on the YOLO-format lists made by prepare.py /
mask_ignore.py (labels found by replacing /images/ -> /labels/), and are scored on all val frames with coco_utils.score.
Frames are pre-resized so the long side is --max-size (default 1280, like imgsz=1280 for YOLO); boxes are mapped back
to source pixels before scoring.

Usage: python src/train_tv.py --model frcnn --train-list data/yolo_masked/train_s3.txt --tag tv_frcnn
"""
import argparse
import functools
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from pycocotools.coco import COCO
from torch import nn
from torchvision.models import detection as D
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.fcos import FCOSClassificationHead
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.transforms.functional import to_tensor

from coco_utils import score
from common import DATA

ap = argparse.ArgumentParser()
ap.add_argument("--model", required=True, choices=["frcnn", "retina", "fcos"])
ap.add_argument("--train-list", required=True)
ap.add_argument("--tag", required=True)
ap.add_argument("--epochs", type=int, default=6)
ap.add_argument("--bs", type=int, default=4)
ap.add_argument("--lr", type=float, default=0.005)
ap.add_argument("--min-size", type=int, default=720)
ap.add_argument("--max-size", type=int, default=1280)
ap.add_argument("--workers", type=int, default=4)
ap.add_argument("--out", default="runs")
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--rpn-post-nms", type=int, default=1000, help="frcnn: proposals kept per image after RPN NMS")
ap.add_argument("--roi-batch", type=int, default=512, help="frcnn: RoIs sampled per image for the box head")
ap.add_argument("--predict-test", action="store_true", help="also save detections for the test set (runs/testdets_<tag>.npy)")
ap.add_argument("--no-val", action="store_true", help="skip scoring on val (e.g. when only test detections are needed)")
ap.add_argument("--viz", type=int, default=0, help="debug: save N augmented training samples with their boxes and exit")
# --- tuning switches (defaults reproduce the baseline exactly) ---
ap.add_argument("--frcnn-beta", type=float, default=1 / 9, help="frcnn: smooth-L1 beta of the box head loss (1e-5 ~ plain L1; torchvision default 1/9)")
ap.add_argument("--anchor-base", type=int, default=32, help="retina: smallest anchor size at P3 (32 = default; 16 reaches the many <32 px objects)")
ap.add_argument("--focal-alpha", type=float, default=0.25, help="retina: focal loss alpha")
ap.add_argument("--focal-gamma", type=float, default=2.0, help="retina: focal loss gamma")
ap.add_argument("--retina-loss", default="l1", choices=["l1", "smooth_l1", "giou", "diou", "ciou"], help="retina: box regression loss")
ap.add_argument("--fcos-radius", type=float, default=1.5, help="fcos: center-sampling radius (positives = locations within radius x stride of the box centre)")
ap.add_argument("--limit-train", type=int, default=0, help="debug: stop after N iterations")
ap.add_argument("--limit-val", type=int, default=0, help="debug: evaluate only the first N frames")
a = ap.parse_args()
torch.manual_seed(a.seed); np.random.seed(a.seed)
dev = "cuda" if torch.cuda.is_available() else "cpu"
NC = 8 + 1  # 8 classes + background (torchvision convention: label 0 = background)


class DS(torch.utils.data.Dataset):
    """Image -> (tensor, target, scale, id). Frames are pre-resized to the working size; hflip only when training."""

    def __init__(s, files, train, ids=None):
        s.files, s.train, s.ids = files, train, ids

    def __len__(s):
        return len(s.files)

    def __getitem__(s, i):
        im = Image.open(s.files[i]).convert("RGB"); W, H = im.size
        sc = min(a.min_size / min(W, H), a.max_size / max(W, H))
        im = im.resize((round(W * sc), round(H * sc)), Image.BILINEAR)
        tgt = {}
        if s.train:
            lab = Path(s.files[i].replace("/images/", "/labels/")).with_suffix(".txt")
            r = np.loadtxt(lab, ndmin=2) if lab.exists() and lab.stat().st_size else np.zeros((0, 5))
            w, h = im.size
            cx, cy, bw, bh = (r[:, 1:5] * np.array([w, h, w, h])).T
            b = np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], 1).reshape(-1, 4)
            keep = ((b[:, 2] - b[:, 0]) >= 1) & ((b[:, 3] - b[:, 1]) >= 1)
            b, c = b[keep], r[keep, 0].astype(np.int64) + 1
            x = to_tensor(im)
            if torch.rand(1).item() < 0.5:  # horizontal flip (torch RNG is re-seeded per dataloader worker, numpy is not)
                x = x.flip(-1); b = b.copy(); b[:, [0, 2]] = w - b[:, [2, 0]]
            tgt = dict(boxes=torch.as_tensor(b, dtype=torch.float32).reshape(-1, 4), labels=torch.as_tensor(c))
            return x, tgt, sc, -1
        return to_tensor(im), tgt, sc, s.ids[i]


def patch_losses():
    """Apply the loss/assignment switches. Each patch is a no-op at its default value."""
    if a.model == "frcnn" and a.frcnn_beta != 1 / 9:
        import torch.nn.functional as F
        from torchvision.models.detection import roi_heads

        def fastrcnn_loss(class_logits, box_regression, labels, regression_targets):  # torchvision's, with a free beta
            labels = torch.cat(labels, dim=0); regression_targets = torch.cat(regression_targets, dim=0)
            cls_loss = F.cross_entropy(class_logits, labels)
            pos = torch.where(labels > 0)[0]; lab = labels[pos]
            N = class_logits.shape[0]
            box_regression = box_regression.reshape(N, box_regression.size(-1) // 4, 4)
            box_loss = F.smooth_l1_loss(box_regression[pos, lab], regression_targets[pos], beta=a.frcnn_beta, reduction="sum")
            return cls_loss, box_loss / labels.numel()
        roi_heads.fastrcnn_loss = fastrcnn_loss
    if a.model == "retina" and (a.focal_alpha, a.focal_gamma) != (0.25, 2.0):
        from torchvision.models.detection import retinanet
        from torchvision.ops import sigmoid_focal_loss
        retinanet.sigmoid_focal_loss = functools.partial(sigmoid_focal_loss, alpha=a.focal_alpha, gamma=a.focal_gamma)


def build():
    """COCO-pretrained detector with a fresh 8-class head; inference thresholds set for mAP (low score, up to 500 dets)."""
    size = dict(min_size=(a.min_size - 80, a.min_size - 40, a.min_size + 40, a.min_size + 80, a.min_size),  # multi-scale train; eval uses the last
                max_size=int(a.max_size * 1.12))
    if a.model == "frcnn":
        m = D.fasterrcnn_resnet50_fpn_v2(weights=D.FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, box_score_thresh=0.001,
                                         box_detections_per_img=500, rpn_post_nms_top_n_train=a.rpn_post_nms,
                                         rpn_post_nms_top_n_test=a.rpn_post_nms, box_batch_size_per_image=a.roi_batch, **size)
        m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NC)
    elif a.model == "retina":
        m = D.retinanet_resnet50_fpn_v2(weights=D.RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1, score_thresh=0.001,
                                        detections_per_img=500, **size)
        m.head.classification_head = RetinaNetClassificationHead(
            256, m.head.classification_head.num_anchors, NC, norm_layer=functools.partial(nn.GroupNorm, 32))
        m.head.regression_head._loss_type = a.retina_loss
        if a.anchor_base != 32:  # same 3 sizes x 3 ratios per location as the default, so the pretrained head still fits
            from torchvision.models.detection.anchor_utils import AnchorGenerator
            sizes = tuple((int(a.anchor_base * 2 ** i), int(a.anchor_base * 2 ** (i + 1 / 3)), int(a.anchor_base * 2 ** (i + 2 / 3))) for i in range(5))
            m.anchor_generator = AnchorGenerator(sizes, ((0.5, 1.0, 2.0),) * 5)
    else:
        m = D.fcos_resnet50_fpn(weights=D.FCOS_ResNet50_FPN_Weights.COCO_V1, score_thresh=0.001,
                                detections_per_img=500, **size)
        m.head.classification_head = FCOSClassificationHead(256, m.head.classification_head.num_anchors, NC)
        m.center_sampling_radius = a.fcos_radius
    return m.to(dev)


files = Path(a.train_list).read_text().split()
loader = torch.utils.data.DataLoader(DS(files, True), batch_size=a.bs, shuffle=True, num_workers=a.workers, drop_last=True,
                                     collate_fn=lambda b: tuple(zip(*b)), persistent_workers=a.workers > 0)
if a.viz:  # sanity check of labels after resize + flip
    from PIL import ImageDraw
    ds = DS(files, True)
    for k in range(a.viz):
        x, t, _, _ = ds[k * 997 % len(ds)]
        im = Image.fromarray((x.permute(1, 2, 0).numpy() * 255).astype("uint8")); d = ImageDraw.Draw(im)
        for b in t["boxes"].tolist():
            d.rectangle(b, outline=(0, 255, 0), width=2)
        im.save(f"{a.out}/viz_{k}.jpg"); print("saved", k, len(t["boxes"]), "boxes", im.size)
    raise SystemExit
patch_losses()
model = build()
opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=a.lr, momentum=0.9, weight_decay=1e-4)
total = a.epochs * len(loader)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda i: min(1, (i + 1) / 500) * (0.5 * (1 + math.cos(math.pi * min(i, total) / total))))  # warmup + cosine
scaler = torch.amp.GradScaler(enabled=dev == "cuda")
out = Path(a.out) / a.tag; out.mkdir(parents=True, exist_ok=True)
start, log = 0, []
if (out / "last.pt").exists():  # resume after an interrupted session
    ck = torch.load(out / "last.pt", map_location=dev)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    start, log = ck["epoch"], ck["log"]
    print("resumed from epoch", start)

it = 0
for ep in range(start, a.epochs):
    model.train(); t0, run = time.time(), []
    for x, tgt, _, _ in loader:
        x = [i.to(dev) for i in x]; tgt = [{k: v.to(dev) for k, v in t.items()} for t in tgt]
        with torch.autocast(dev, enabled=dev == "cuda"):
            losses = model(x, tgt)
            loss = sum(losses.values())
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 10.0)  # RetinaNet/FCOS can spike early with AMP
        scaler.step(opt); scaler.update(); sched.step()
        run.append({k: v.item() for k, v in losses.items()}); it += 1
        if it % 100 == 0:
            print(f"ep {ep + 1}/{a.epochs} it {it} loss {loss.item():.3f} lr {sched.get_last_lr()[0]:.5f} "
                  f"{(time.time() - t0) / len(run):.2f}s/it", flush=True)
        if a.limit_train and it >= a.limit_train:
            break
    log.append(dict(epoch=ep + 1, minutes=(time.time() - t0) / 60, **{k: float(np.mean([r[k] for r in run])) for k in run[0]}))
    print(log[-1], flush=True)
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=ep + 1, log=log), out / "last.pt")
    if a.limit_train and it >= a.limit_train:
        break
json.dump(log, open(out / "train_log.json", "w"), indent=1)

# ---- inference helpers + evaluation on val with the project protocol ----------------------------------------------
def infer(paths, ids):
    """Detections [dict(image_id, category_id, score, bbox xywh in source pixels)] for the given image files."""
    dl = torch.utils.data.DataLoader(DS(paths, False, ids), batch_size=4, num_workers=a.workers,
                                     collate_fn=lambda b: tuple(zip(*b)))
    out = []
    model.eval()
    with torch.no_grad(), torch.autocast(dev, enabled=dev == "cuda"):
        for x, _, scs, iids in dl:
            for o, sc, iid in zip(model([i.to(dev) for i in x]), scs, iids):
                for (x1, y1, x2, y2), p, c in zip((o["boxes"] / sc).tolist(), o["scores"].tolist(), o["labels"].tolist()):
                    out.append(dict(image_id=iid, category_id=c, score=p, bbox=[x1, y1, x2 - x1, y2 - y1]))
    return out


if a.predict_test:  # image_id = row index in test_images.csv (ensemble.py --split test maps it back to the name)
    import pandas as pd
    t = pd.read_csv(DATA / "test_images.csv")
    d = infer([str(DATA / f) for f in t.file_name], list(range(len(t))))
    np.save(Path(a.out) / f"testdets_{a.tag}.npy", np.array([[x["image_id"], x["category_id"], x["score"], *x["bbox"]] for x in d], np.float32))
    print("saved test detections:", len(d))
if not a.no_val:
    gt = COCO(str(DATA / "val" / "annotations_coco.json"))
    ids = sorted(gt.getImgIds())[: a.limit_val or None]
    t0 = time.time()
    dets = infer([str(DATA / "val" / gt.imgs[i]["file_name"]) for i in ids], ids)
    ms = (time.time() - t0) / len(ids) * 1000
    score(gt, dets, ids, a.tag, ms, out_dir=a.out, **{k: v for k, v in vars(a).items() if k not in ("train_list", "out", "tag")})

Writing src/train_tv.py


### `src/ensemble.py`
Fuses serialized detection arrays (`dets_*.npy` / `testdets_*.npy`) using Weighted Boxes Fusion (WBF), parallelized across CPU cores with `multiprocessing.Pool`.

In [10]:
%%writefile src/ensemble.py
"""Fuse saved detections of several models with Weighted Boxes Fusion (multiprocessed across CPU cores) and score on val or write test CSV.

Based on temp/src/ensemble.py (Pool(workers) for fast parallel fusion across 2,846 val / 3,811 test frames).
"""
import argparse
import contextlib
import io
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
from ensemble_boxes import weighted_boxes_fusion
try:
    import faster_coco_eval
    faster_coco_eval.init_as_pycocotools()
except Exception:
    pass
from pycocotools.coco import COCO

from coco_utils import score
from common import DATA

ap = argparse.ArgumentParser()
ap.add_argument("--tags", required=True, help="comma-separated run tags (<dir>/dets_<tag>.npy)")
ap.add_argument("--weights", default="", help="comma-separated model weights (default all 1)")
ap.add_argument("--iou", type=float, default=0.7, help="WBF IoU threshold for merging boxes")
ap.add_argument("--skip", type=float, default=0.001, help="drop boxes below this score before fusion")
ap.add_argument("--conf-type", default="avg", choices=["avg", "max", "box_and_model_avg", "absent_model_aware_avg"])
ap.add_argument("--tag", default="ens")
ap.add_argument("--dir", default="runs", help="directory holding dets_<tag>.npy / testdets_<tag>.npy")
ap.add_argument("--split", default="val", choices=["val", "test"])
ap.add_argument("--out", default="submission.csv", help="test mode: Kaggle CSV to write")
ap.add_argument("--max-det", type=int, default=500, help="keep top-N fused boxes per image (300 for Kaggle submission)")
ap.add_argument("--workers", type=int, default=8)
a = ap.parse_args()

dpath = Path(a.dir)
tags = [t.strip() for t in a.tags.split(",") if t.strip()]
w = [float(x) for x in a.weights.split(",")] if a.weights else [1.0] * len(tags)
if a.split == "val":
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(DATA / "val" / "annotations_coco.json"))
    img_ids = sorted(gt.getImgIds())
    size = {i: (gt.imgs[i]["width"], gt.imgs[i]["height"]) for i in img_ids}
    PREFIX = "dets"
else:
    test = pd.read_csv(DATA / "test_images.csv")
    img_ids = list(range(len(test)))
    size = {i: (int(r.width), int(r.height)) for i, r in enumerate(test.itertuples())}
    PREFIX = "testdets"


def per_image(tag):
    D = np.load(dpath / f"{PREFIX}_{tag}.npy")
    D = D[np.argsort(D[:, 0], kind="stable")]
    ids, start = np.unique(D[:, 0], return_index=True)
    return {int(i): c for i, c in zip(ids, np.split(D, start[1:]))}


MODELS = [per_image(t) for t in tags]


def fuse(iid):
    W, H = size[iid]
    bl, sl, ll = [], [], []
    for m in MODELS:
        d = m.get(iid)
        if d is None or len(d) == 0:
            bl.append(np.zeros((0, 4))); sl.append(np.zeros(0)); ll.append(np.zeros(0)); continue
        b = np.stack([d[:, 3], d[:, 4], d[:, 3] + d[:, 5], d[:, 4] + d[:, 6]], 1) / [W, H, W, H]
        bl.append(np.clip(b, 0, 1)); sl.append(d[:, 2]); ll.append(d[:, 1] - 1)
    if sum(len(s) for s in sl) == 0:
        return []
    b, s, l = weighted_boxes_fusion(bl, sl, ll, weights=w, iou_thr=a.iou, skip_box_thr=a.skip, conf_type=a.conf_type)
    keep = np.argsort(-s)[: a.max_det]
    b = b[keep] * [W, H, W, H]
    return [dict(image_id=iid, category_id=int(c) + 1, score=float(p), bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)])
            for (x1, y1, x2, y2), p, c in zip(b, s[keep], l[keep])]


if __name__ == "__main__":
    t0 = time.time()
    with Pool(a.workers) as p:
        out = p.map(fuse, img_ids, chunksize=16)
    ms = (time.time() - t0) / len(img_ids) * 1000
    if a.split == "val":
        score(gt, [d for o in out for d in o], img_ids, a.tag, ms, out_dir=str(dpath),
              tags=a.tags, weights=w, wbf_iou=a.iou, skip=a.skip, conf_type=a.conf_type, max_det=a.max_det)
    else:
        rows = [(test.image_id[i], " ".join(f"{d['category_id'] - 1} {d['score']:.4f} {d['bbox'][0]:.1f} {d['bbox'][1]:.1f} "
                                            f"{d['bbox'][2]:.1f} {d['bbox'][3]:.1f}" for d in o)) for i, o in zip(img_ids, out)]
        pd.DataFrame(rows, columns=["image_id", "PredictionString"]).to_csv(a.out, index=False)
        print(f"wrote {a.out}: {len(rows)} rows, {sum(len(o) for o in out) / len(out):.1f} dets/image ({ms:.1f} ms/frame)")

Writing src/ensemble.py


### `src/predict_test.py`
Runs batch inference on the test video frames and exports predictions to `testdets_<tag>.npy` for rapid WBF merging.

In [11]:
%%writefile src/predict_test.py
"""Save an Ultralytics model's detections for the TEST set as <out>/testdets_<tag>.npy
(rows: test_images.csv row index, category 1-based, score, x, y, w, h in source pixels).
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from ultralytics import RTDETR, YOLO

from common import DATA

ap = argparse.ArgumentParser()
ap.add_argument("--weights", required=True)
ap.add_argument("--imgsz", type=int, default=1600)
ap.add_argument("--conf", type=float, default=0.001)
ap.add_argument("--iou", type=float, default=0.6)
ap.add_argument("--bs", type=int, default=8)
ap.add_argument("--multi-label", action="store_true")
ap.add_argument("--tta", action="store_true")
ap.add_argument("--arch", default="yolo", choices=["yolo", "rtdetr"])
ap.add_argument("--tag", required=True)
ap.add_argument("--out", default="runs")
a = ap.parse_args()
if a.multi_label:
    import functools
    from ultralytics.utils import nms
    nms.non_max_suppression = functools.partial(nms.non_max_suppression, multi_label=True)

t = pd.read_csv(DATA / "test_images.csv")
files = [str(DATA / f) for f in t.file_name]
model, rows = (RTDETR if a.arch == "rtdetr" else YOLO)(a.weights), []
for s in range(0, len(files), a.bs):
    for r, i in zip(model.predict(files[s:s + a.bs], imgsz=a.imgsz, conf=a.conf, iou=a.iou, max_det=500, batch=a.bs,
                                  augment=a.tta, verbose=False), range(s, s + a.bs)):
        xyxy, sc, c = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()
        rows += [[i, int(k) + 1, p, x1, y1, x2 - x1, y2 - y1] for (x1, y1, x2, y2), p, k in zip(xyxy, sc, c)]
dst = Path(a.out) / f"testdets_{a.tag}.npy"
np.save(dst, np.array(rows, np.float32))
print("saved", len(rows), "detections for", len(files), "test frames ->", dst)

Writing src/predict_test.py


## Step 1: Dataset Preparation

We prepare two dataset variants on local SSD:
1. **`yolo_masked`**: 56 training sequences with stride 3 (8,090 frames) with ignore regions masked. Used for training validation models.
2. **`yolo_masked_tv`**: All 63 sequences (`train + val`, 9,041 frames). To save time, already-masked training frames are symlinked from `yolo_masked`. Used for final test model retraining.

In [12]:
!python src/prepare.py --stride 3 --out {WORK}/yolo
!python src/mask_ignore.py --stride 3 --out {WORK}/yolo_masked
!python src/mask_ignore.py --stride 3 --splits train val --reuse {WORK}/yolo_masked --out {WORK}/yolo_masked_tv

train 8090 frames (stride 3), val 2846 frames -> /content/yolo
8090 frames (train), 6053 newly masked, 4115 labels dropped -> /content/yolo_masked
9041 frames (train+val), 480 newly masked, 413 labels dropped -> /content/yolo_masked_tv


## Step 2: Multi-Model Training

### Models Trained & Architecture Rationale:
* **`s1280_dfl3` (`yolo11s`, 1280px)**: Compact backbone providing high-recall bounding box candidates across dense small-scale regions.
* **`l1280_dfl3` (`yolo11l`, 1280px)**: Large backbone with deep feature hierarchy, improving classification accuracy on heavily occluded vehicles.
* **`m1600_dfl3` (`yolo11m`, 1600px)**: Trained natively at 1600px image size. Higher input resolution provides vital pixel density for tiny drone targets (often < 16 px).
* **`tv_y_dfl3` (`yolo11m`, 1280px, `train+val`)**: Retrained on all 9,041 `train+val` frames to learn background scenes and lighting conditions from validation clips.
* **`tv_l_dfl3` (`yolo11l`, 1280px, `train+val`)**: High-capacity large model retrained on all 9,041 frames for maximum test set performance.

### Training Configuration:
* **`dfl=3.0`**: Increased Distribution Focal Loss weight (from 1.5 default) forces tighter box boundary distributions on blurry drone targets.
* **`cache=False`**: Reads directly from fast local NVMe SSD to avoid high system RAM usage and prevent OOM issues.
* **Execution**: Runs in parallel across worker slots; completed models in Google Drive are automatically skipped.

In [14]:
import os, pathlib, subprocess, time
from concurrent.futures import ThreadPoolExecutor

P, W = 4, 3          # 4 parallel jobs x 3 workers = 12 vCPUs; ~29 GB / 80 GB VRAM; ~72 GB / 167 GB RAM
COMMON = f"epochs=6 workers={W} cache=False optimizer=SGD lr0=0.01 mixup=0.15 scale=0.9 close_mosaic=0 dfl=3.0 seed=0 deterministic=True project={OUT} exist_ok=True plots=False"
D_TR = f"{WORK}/yolo_masked/data_s3.yaml"
D_TV = f"{WORK}/yolo_masked_tv/data_trainval_s3.yaml"

# s1280_dfl3 is placed first so it finishes in ~20 min and hands its slot to tv_l_dfl3 (train+val yolo11l)
TRAIN_JOBS = {
    "s1280_dfl3": f"yolo detect train model=yolo11s.pt data={D_TR} imgsz=1280 batch=8 {COMMON} name=s1280_dfl3",
    "l1280_dfl3": f"yolo detect train model=yolo11l.pt data={D_TR} imgsz=1280 batch=4 {COMMON} name=l1280_dfl3",
    "m1600_dfl3": f"yolo detect train model=yolo11m.pt data={D_TR} imgsz=1600 batch=4 {COMMON} name=m1600_dfl3",
    "tv_y_dfl3":  f"yolo detect train model=yolo11m.pt data={D_TV} imgsz=1280 batch=4 {COMMON} name=tv_y_dfl3",
    "tv_l_dfl3":  f"yolo detect train model=yolo11l.pt data={D_TV} imgsz=1280 batch=4 {COMMON} name=tv_l_dfl3",
}

def run_train(item):
    name, cmd = item
    if (pathlib.Path(OUT) / name / "weights/last.pt").exists():
        return name, "skipped (weights/last.pt exists)", 0.0
    t0 = time.time()
    logs = pathlib.Path(OUT) / "logs"; logs.mkdir(exist_ok=True)
    with open(logs / f"{name}.log", "w") as f:
        rc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT).returncode
    return name, "ok" if rc == 0 else f"FAILED rc={rc}", round((time.time() - t0) / 60, 1)

with ThreadPoolExecutor(P) as ex:
    for r in ex.map(run_train, TRAIN_JOBS.items()):
        print(r, flush=True)

('s1280_dfl3', 'ok', 56.6)
('l1280_dfl3', 'ok', 84.3)
('m1600_dfl3', 'ok', 88.1)
('tv_y_dfl3', 'ok', 98.2)
('tv_l_dfl3', 'ok', 128.6)


## Step 3: Multi-Scale & Multi-Label Inference

### Inference Configuration:
* **High-Resolution Evaluation**: Models are evaluated at 1600px and 1920px image sizes to resolve tiny objects.
* **Test-Time Augmentation (TTA)**: Multi-scale testing and horizontal flips are applied to top models to stabilize box boundaries.
* **Single-Label (`sl`) vs. Multi-Label (`ml`)**:
  * **Single-Label (`sl`)**: Standard argmax prediction (one class per box). Keeps every detection slot focused on distinct physical objects.
  * **Multi-Label (`ml`)**: Retains multiple class hypotheses per box above threshold. Helps resolve VisDrone visually confusable categories (car vs. van, van vs. truck).
* **Caching**: Checkpoints are copied to local SSD (`/content/w_cache`) to prevent Google Drive I/O bottlenecks. Detections are saved to `.npy` files (`dets_*.npy` for val, `testdets_*.npy` for test).

In [17]:
import pathlib, shutil, subprocess, time
from concurrent.futures import ThreadPoolExecutor

# Ensure faster_coco_eval is initialized BEFORE pycocotools.coco is imported in eval_coco.py
ev_path = pathlib.Path("src/eval_coco.py")
ev_txt = ev_path.read_text()
if "faster_coco_eval" not in ev_txt:
    ev_path.write_text(
        "try:\n    import faster_coco_eval; faster_coco_eval.init_as_pycocotools()\nexcept Exception:\n    pass\n" + ev_txt
    )

# If any finished val run already sits in local runs/, copy it to OUT right away
for f in pathlib.Path("runs").glob("dets_*.npy"):
    tag = f.stem[len("dets_"):]
    ev = pathlib.Path("runs") / f"eval_{tag}.json"
    if ev.exists() and not (pathlib.Path(OUT) / f.name).exists():
        shutil.copy2(f, pathlib.Path(OUT) / f.name)
        shutil.copy2(ev, pathlib.Path(OUT) / ev.name)

VAL_AND_TEST = [
    # Existing yolo11m members from experiments.ipynb (already in OUT on your Drive -> skipped in 0s)
    ("yolo_dfl3_s_i1600_tta",  f"{OUT}/yolo_dfl3/weights/best.pt",  1600, "--tta"),
    ("yolo_dfl3_ml_tta",       f"{OUT}/yolo_dfl3/weights/best.pt",  1600, "--multi-label --tta"),
    ("yolo_box12_s_i1600_tta", f"{OUT}/yolo_box12/weights/best.pt", 1600, "--tta"),
    ("yolo_box12_ml",          f"{OUT}/yolo_box12/weights/best.pt", 1600, "--multi-label"),
    # New scale/backbone members trained in Step 2 (single-label = e3_d in temp/src/run_ens3.sh)
    ("l1280_dfl3_i1600",       f"{OUT}/l1280_dfl3/weights/best.pt", 1600, ""),
    ("m1600_dfl3_i1920",       f"{OUT}/m1600_dfl3/weights/best.pt", 1920, ""),
    ("s1280_dfl3_i1280",       f"{OUT}/s1280_dfl3/weights/best.pt", 1280, ""),
    # New scale/backbone members trained in Step 2 (multi-label = temp/src/run_ml2.sh)
    ("l1280_dfl3_ml",          f"{OUT}/l1280_dfl3/weights/best.pt", 1600, "--multi-label"),
    ("m1600_dfl3_ml",          f"{OUT}/m1600_dfl3/weights/best.pt", 1920, "--multi-label"),
    ("s1280_dfl3_ml",          f"{OUT}/s1280_dfl3/weights/best.pt", 1280, "--multi-label"),
]

TEST_ONLY_TV = [
    # Train+val models (temp/src/run_final.sh: use last.pt for test predictions)
    ("tv_y_dfl3_tta",          f"{OUT}/tv_y_dfl3/weights/last.pt",  1600, "--tta"),
    ("tv_y_dfl3_ml_tta",       f"{OUT}/tv_y_dfl3/weights/last.pt",  1600, "--multi-label --tta"),
    ("tv_l_dfl3_i1600",        f"{OUT}/tv_l_dfl3/weights/last.pt",  1600, ""),
    ("tv_l_dfl3_ml",           f"{OUT}/tv_l_dfl3/weights/last.pt",  1600, "--multi-label"),
]

# Cache weight files to local NVMe disk (/content/w_cache) to prevent Google Drive FUSE read errors
w_cache = pathlib.Path(WORK) / "w_cache"; w_cache.mkdir(exist_ok=True)
def local_w(w_path):
    p = pathlib.Path(w_path)
    if not p.exists() and (p.parent / "last.pt").exists():
        p = p.parent / "last.pt"
    if not p.exists() and (p.parent / "best.pt").exists():
        p = p.parent / "best.pt"
    if not p.exists():
        return str(p)
    dst = w_cache / f"{p.parent.parent.name}_{p.name}"
    if not dst.exists() or dst.stat().st_size != p.stat().st_size:
        shutil.copy2(p, dst)
    return str(dst)

tasks = []
for tag, w_path, sz, extra in VAL_AND_TEST:
    lw = local_w(w_path)
    if not (pathlib.Path(OUT) / f"dets_{tag}.npy").exists():
        tasks.append((f"val  {tag}", f"python src/eval_coco.py --weights {lw} --imgsz {sz} --bs 8 {extra} --tag {tag} && cp runs/eval_{tag}.json runs/dets_{tag}.npy {OUT}/"))
    if not (pathlib.Path(OUT) / f"testdets_{tag}.npy").exists():
        tasks.append((f"test {tag}", f"python src/predict_test.py --weights {lw} --imgsz {sz} --bs 8 {extra} --tag {tag} --out {OUT}"))

for tag, w_path, sz, extra in TEST_ONLY_TV:
    lw = local_w(w_path)
    if not (pathlib.Path(OUT) / f"testdets_{tag}.npy").exists():
        tasks.append((f"test {tag}", f"python src/predict_test.py --weights {lw} --imgsz {sz} --bs 8 {extra} --tag {tag} --out {OUT}"))

print(f"Running {len(tasks)} remaining inference tasks across 2 GPU streams (--bs 8)...")
failed = []
def run_inf(item):
    label, cmd = item
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    dt = round(time.time() - t0, 1)
    if p.returncode == 0:
        return label, "ok", dt, ""
    err = "\n".join(p.stdout.splitlines()[-6:] + p.stderr.splitlines()[-6:])
    return label, f"FAILED rc={p.returncode}", dt, err

with ThreadPoolExecutor(2) as ex:
    for label, status, dt, err in ex.map(run_inf, tasks):
        print((label, status, dt), flush=True)
        if status != "ok":
            print(f"  -> Error for {label}:\n{err}", flush=True)
            failed.append((label, dict(tasks)[label]))

# Sequential fallback with --bs 4 if anything failed
for label, cmd in failed:
    print(f"Retrying {label} sequentially with --bs 4...", flush=True)
    t0 = time.time()
    p = subprocess.run(cmd.replace("--bs 8", "--bs 4"), shell=True, capture_output=True, text=True)
    print((label, "ok (retry)" if p.returncode == 0 else f"FAILED rc={p.returncode}", round(time.time() - t0, 1)), flush=True)
    if p.returncode != 0:
        print(p.stderr[-800:], flush=True)

Running 10 remaining inference tasks across 2 GPU streams (--bs 8)...
('val  l1280_dfl3_i1600', 'ok', 224.8)
('test l1280_dfl3_i1600', 'ok', 151.0)
('val  m1600_dfl3_i1920', 'ok', 214.1)
('val  s1280_dfl3_i1280', 'ok', 174.9)
('val  l1280_dfl3_ml', 'ok', 218.3)
('test l1280_dfl3_ml', 'ok', 148.7)
('val  m1600_dfl3_ml', 'ok', 232.2)
('test m1600_dfl3_ml', 'ok', 160.4)
('test tv_l_dfl3_i1600', 'ok', 167.1)
('test tv_l_dfl3_ml', 'ok', 165.5)


## Step 4: Weighted Boxes Fusion (WBF) & Final Submissions

### Ensembling Strategy:
* **Weighted Boxes Fusion (WBF) vs. NMS**: Standard NMS discards overlapping boxes from different models. WBF computes confidence-weighted coordinate averages across overlapping boxes, producing tighter and more accurate bounding boxes.
* **Model Selection & Weights**:
  * **Validation Ensemble (8 models, weights `2, 2, 2, 1, 1, 1, 1, 1`)**:
    - Weight 2.0: Core high-performing YOLO models (`yolo_dfl3` TTA @ 1600px, `l1280_dfl3` @ 1600px, `m1600_dfl3` @ 1920px).
    - Weight 1.0: Diverse complementary detectors (`yolo_box12`, `s1280_dfl3`, Faster R-CNN, RetinaNet, FCOS).
  * **Test Ensemble (10 models, weights `2, 2, 2, 2, 2, 1, 1, 1, 1, 1`)**: Adds `train+val` models `tv_y_dfl3` and `tv_l_dfl3` (weight 2.0 each).
* **Parameters**:
  * **IoU Threshold = 0.7**: Fuses true duplicate detections while preserving closely spaced distinct vehicles in dense traffic.
  * **Max Detections = 300**: Keeps the top 300 highest-confidence boxes per frame, staying well within Kaggle CSV file size limits (~35 MB) while capturing over 99% of ground-truth objects.

### Submission Files:
1. **`sub_e3d_dfl3_trainval_ml_top300.csv`**: Multi-label fusion (**0.3264 Val mAP**, **0.3774 AP_no4k**).
2. **`sub_e3d_dfl3_trainval_sl_top300.csv`**: Single-label fusion (**0.3217 Val mAP**, safe conservative submission).

In [18]:
import json, pathlib, subprocess
import pandas as pd

def run_wbf(split, tags_csv, weights_csv, tag_or_out):
    prefix = "dets" if split == "val" else "testdets"
    tags = [t.strip() for t in tags_csv.split(",")]
    ws = [w.strip() for w in weights_csv.split(",")]
    kept_t, kept_w, missing = [], [], []
    for t, w in zip(tags, ws):
        if (pathlib.Path(OUT) / f"{prefix}_{t}.npy").exists():
            kept_t.append(t); kept_w.append(w)
        else:
            missing.append(t)
    if missing:
        print(f"[WARN] {tag_or_out}: skipping missing {prefix} for {missing} (fusing {len(kept_t)}/{len(tags)} models)", flush=True)
    else:
        print(f"[OK] {tag_or_out}: fusing all {len(kept_t)}/{len(tags)} models", flush=True)
    extra = f"--tag {tag_or_out}" if split == "val" else f"--out {tag_or_out}"
    cmd = (f"python src/ensemble.py --workers 12 --split {split} --dir {OUT} "
           f"--tags {','.join(kept_t)} --weights {','.join(kept_w)} --iou 0.7 --max-det 300 {extra}")
    subprocess.run(cmd, shell=True, check=False)

SL_VAL = "yolo_dfl3_s_i1600_tta,l1280_dfl3_i1600,m1600_dfl3_i1920,tv_frcnn_beta1,tv_retina_giou,tv_fcos_r10,yolo_box12_s_i1600_tta,s1280_dfl3_i1280"
ML_VAL = "yolo_dfl3_ml_tta,l1280_dfl3_ml,m1600_dfl3_ml,tv_frcnn_beta1,tv_retina_giou,tv_fcos_r10,yolo_box12_ml,s1280_dfl3_ml"
W8 = "2,2,2,1,1,1,1,1"

run_wbf("val", SL_VAL, W8, "e3d_dfl3_sl_300")
run_wbf("val", ML_VAL, W8, "e3d_dfl3_ml_300")

# 2. Test submissions (adding the train+val models tv_y_dfl3 and tv_l_dfl3 from temp/src/run_final.sh)
SL_TEST = f"tv_y_dfl3_tta,tv_l_dfl3_i1600,{SL_VAL}"
ML_TEST = f"tv_y_dfl3_ml_tta,tv_l_dfl3_ml,{ML_VAL}"
W10 = "2,2,2,2,2,1,1,1,1,1"

run_wbf("test", SL_TEST, W10, f"{OUT}/sub_e3d_dfl3_trainval_sl_top300.csv")
run_wbf("test", ML_TEST, W10, f"{OUT}/sub_e3d_dfl3_trainval_ml_top300.csv")

# 3. Full summary table sorted by AP
rows = [json.load(open(f)) for f in sorted(pathlib.Path(OUT).glob("eval_*.json"))]
cols = ["tag", "AP", "AP_no4k", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_frame"]
pd.DataFrame([{k: r.get(k) for k in cols} for r in rows]).round(4).sort_values("AP", ascending=False).reset_index(drop=True)

[OK] e3d_dfl3_sl_300: fusing all 8/8 models
[OK] e3d_dfl3_ml_300: fusing all 8/8 models
[OK] /content/drive/MyDrive/hw2_runs/sub_e3d_dfl3_trainval_sl_top300.csv: fusing all 10/10 models
[OK] /content/drive/MyDrive/hw2_runs/sub_e3d_dfl3_trainval_ml_top300.csv: fusing all 10/10 models


,tag,AP,AP_no4k,AP50,AP_small,AP_medium,AP_large,ms_per_frame
0,e3d_dfl3_ml_300,0.3264,0.3774,0.5917,0.1764,0.2943,0.5027,33.0345
1,e3d_dfl3_sl_300,0.3217,0.3684,0.5862,0.1772,0.2848,0.5000,32.2694
2,ens_5_all_iou70,0.3078,0.3539,0.5680,0.1630,0.2785,0.4801,139.3686
3,ens_7_fix_singlelabel_500,0.3074,0.3530,0.5668,0.1583,0.2787,0.4887,151.3032
4,ens_6_all_iou70_top300,0.3062,0.3524,0.5647,0.1622,0.2771,0.4782,141.4885
5,ens_8_fix_singlelabel_300,0.3058,0.3513,0.5633,0.1575,0.2772,0.4871,152.5419
6,ens_3_yolo_frcnn,0.3038,0.3508,0.5636,0.1656,0.2776,0.4689,90.3302
7,ens_4_all_iou60,0.3012,0.3456,0.5662,0.1582,0.2724,0.4692,151.3883
8,ens_9_fix_temporal_300,0.3008,0.3475,0.5491,0.1561,0.2710,0.4808,4.4786
9,l1280_dfl3_ml,0.2945,0.3424,0.5536,0.1381,0.3182,0.4197,66.8973
